# Notebook 04: Social Enrichment

**Purpose:** Add LinkedIn, Twitter, and GitHub profiles to researchers

**Input:** `data/researchers.csv`

**Output:** `data/researchers_enriched.csv`

This notebook uses DuckDuckGo search to find social media profiles for each researcher.

## Setup

In [ ]:
# Import required libraries
import pandas as pd
import time
import re
import urllib.parse
from tqdm import tqdm
from ddgs import DDGS

# Add project root to path for imports
import sys
from pathlib import Path
project_root = Path.cwd().parent
sys.path.append(str(project_root))

# Import our utilities
from scripts.utils import load_config, ensure_data_dir, save_csv_checkpoint, load_csv_checkpoint

# Load configuration
config = load_config()
print(f"Loaded config for domain: {config['domain']['name']}")

# Set up data directory
data_dir = ensure_data_dir(config['output']['data_dir'])

## Load Researcher Data

In [ ]:
# Load researchers from previous step
researchers_df = load_csv_checkpoint("researchers.csv")
if researchers_df is None:
    raise FileNotFoundError("researchers.csv not found. Please run notebook 03 first.")

print(f"Loaded {len(researchers_df)} researchers")
researchers_df.head()

## Social Profile Search Functions

In [ ]:
def is_valid_profile_url(url: str, platform: str) -> bool:
    """
    Validate if URL is a plausible profile for the given platform
    """
    if not url or not isinstance(url, str):
        return False
    
    url_lower = url.lower()
    
    # Platform-specific validation
    if platform == 'linkedin':
        return ('linkedin.com/in/' in url_lower or 
                'linkedin.com/pub/' in url_lower) and \
               not any(skip in url_lower for skip in ['company', 'school', 'jobs', 'search'])
    
    elif platform == 'twitter':
        return ('twitter.com/' in url_lower or 
                'x.com/' in url_lower) and \
               not any(skip in url_lower for skip in ['explore', 'search', 'status', 'home'])
    
    elif platform == 'github':
        return 'github.com/' in url_lower and \
               not any(skip in url_lower for skip in ['orgs', 'topics', 'search', 'marketplace'])
    
    return False

def search_social_profile(name: str, affiliation: str, platform: str, max_results: int = 5) -> str:
    """
    Search for a researcher's social profile using DuckDuckGo
    
    Returns the first valid profile URL found, or empty string if none found
    """
    # Build search query
    affiliation_clean = affiliation.replace("Unknown", "").strip()
    if affiliation_clean:
        query = f'"{name}" "{affiliation_clean}" {platform}'
    else:
        query = f'"{name}" {platform} researcher'
    
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=max_results))
            
            # Look for the first valid profile URL
            for result in results:
                url = result.get('href', '')
                if is_valid_profile_url(url, platform):
                    return url
                    
    except Exception as e:
        print(f"Search failed for {name} on {platform}: {e}")
    
    return ""

def find_researcher_profiles(row):
    """
    Find social profiles for a single researcher
    """
    name = row['name']
    affiliation = row['affiliation']
    
    # Rate limiting: small delay between searches
    time.sleep(0.5)
    
    linkedin = search_social_profile(name, affiliation, 'linkedin')
    time.sleep(0.5)
    
    twitter = search_social_profile(name, affiliation, 'twitter')
    time.sleep(0.5)
    
    github = search_social_profile(name, affiliation, 'github')
    
    return pd.Series({
        'linkedin': linkedin,
        'twitter': twitter,
        'github': github
    })

# Test the search function
print("Testing social profile search...")
test_result = search_social_profile("Andrew Ng", "Stanford University", "linkedin")
print(f"Test search result: {test_result}")

## Process Researchers in Batches

In [ ]:
# Set processing limits from config
max_researchers = config['processing'].get('max_researchers', len(researchers_df))
batch_size = config['processing'].get('batch_size', 10)

# Limit to configured max
researchers_to_process = researchers_df.head(max_researchers).copy()
print(f"Processing {len(researchers_to_process)} researchers (batch size: {batch_size})")

# Initialize progress tracking
results = []

# Process in batches
for i in tqdm(range(0, len(researchers_to_process), batch_size), desc="Processing batches"):
    batch = researchers_to_process.iloc[i:i+batch_size]
    
    # Apply social profile search to batch
    batch_results = batch.apply(find_researcher_profiles, axis=1)
    
    # Combine with original data
    batch_with_profiles = pd.concat([batch.reset_index(drop=True), batch_results.reset_index(drop=True)], axis=1)
    results.append(batch_with_profiles)
    
    # Save intermediate checkpoint every 5 batches
    if (i // batch_size + 1) % 5 == 0:
        intermediate_df = pd.concat(results, ignore_index=True)
        save_csv_checkpoint(intermediate_df, "researchers_enriched_checkpoint.csv")
        print(f"\nSaved intermediate checkpoint after {len(intermediate_df)} researchers")

# Combine all results
enriched_df = pd.concat(results, ignore_index=True)

print(f"\nCompleted processing {len(enriched_df)} researchers")
print(f"Found profiles: LinkedIn: {(enriched_df['linkedin'] != '').sum()}, "
      f"Twitter: {(enriched_df['twitter'] != '').sum()}, "
      f"GitHub: {(enriched_df['github'] != '').sum()}")

## Review and Save Results

In [ ]:
# Show sample results
print("Sample enriched researchers:")
enriched_df[['name', 'affiliation', 'linkedin', 'twitter', 'github']].head(10)

In [ ]:
# Save final results
save_csv_checkpoint(enriched_df, "researchers_enriched.csv")
print(f"\nSaved {len(enriched_df)} enriched researcher profiles to data/researchers_enriched.csv")

# Summary statistics
total_profiles = len(enriched_df)
linkedin_found = (enriched_df['linkedin'] != '').sum()
twitter_found = (enriched_df['twitter'] != '').sum()
github_found = (enriched_df['github'] != '').sum()
any_profile_found = ((enriched_df['linkedin'] != '') | (enriched_df['twitter'] != '') | (enriched_df['github'] != '')).sum()

print("\nEnrichment Summary:")
print(f"Total researchers: {total_profiles}")
print(f"LinkedIn profiles found: {linkedin_found} ({linkedin_found/total_profiles*100:.1f}%)")
print(f"Twitter profiles found: {twitter_found} ({twitter_found/total_profiles*100:.1f}%)")
print(f"GitHub profiles found: {github_found} ({github_found/total_profiles*100:.1f}%)")
print(f"Researchers with at least one profile: {any_profile_found} ({any_profile_found/total_profiles*100:.1f}%)")

## Next Steps

The enriched researcher data is now ready for message personalization in **Notebook 05**.